# Galaxy Image Simulation — Part 2: Detection, Fitting, and Analysis
## From pixels to physical parameters

**This is Part 2.** Run `galaxy_image_simulation_part1.ipynb` first for the theory.

**Simulator:** GalSim — [https://galsim-developers.github.io/GalSim/_build/html/index.html](https://galsim-developers.github.io/GalSim/_build/html/index.html)
**Detection & fitting:** photutils — [https://photutils.readthedocs.io/en/stable/](https://photutils.readthedocs.io/en/stable/)
**Reference (GalSim):** Rowe et al. (2015), A&C 10, 121 — [arXiv:1407.7676](https://arxiv.org/abs/1407.7676)
**Reference (GALFIT):** Peng et al. (2002), AJ 124, 266 — [arXiv:astro-ph/0204182](https://arxiv.org/abs/astro-ph/0204182)

---

## Sections in this notebook

| Section | Topic |
|---------|-------|
| 6 | Setup: imports and simulation parameters |
| 9 | Simulating a single galaxy |
| 10 | Simulating a multi-galaxy field |
| 11 | Source detection with photutils |
| 12 | Galaxy profile fitting (Sérsic + Levenberg-Marquardt) |
| 13 | Input–output comparison |
| 14 | Effect of SNR on parameter recovery |
| 15 | Visualising the PSF effect |
| 16 | Professional tools: GALFIT and GALFITools |
| 17 | Summary |
| 18 | Exercises |

---


## 6. Setup and Imports

Identical to Part 1 — run this before any other cell.

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Core scientific stack ────────────────────────────────────────────────────
import numpy as np
from numpy.random import default_rng
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.optimize import curve_fit
from scipy.special import gamma as gamma_func, gammainc

# ── Astronomy ────────────────────────────────────────────────────────────────
import astropy
import astropy.units as u
from astropy.modeling import models, fitting
from astropy.convolution import Gaussian2DKernel, convolve_fft
from astropy.nddata import Cutout2D
from astropy.table import Table

# ── Galaxy image simulator ────────────────────────────────────────────────────
import galsim

# ── Source detection and photometry ──────────────────────────────────────────
import photutils
from photutils.background import Background2D, MedianBackground
from photutils.segmentation import detect_sources, SourceCatalog
from photutils.aperture import CircularAperture

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.labelsize': 12,
    'legend.fontsize': 9,
    'image.origin': 'lower',
})

RNG = default_rng(42)  # reproducible random numbers

print('Imports successful!')
print(f'astropy  {astropy.__version__}')
print(f'photutils {photutils.__version__}')
print(f'galsim   {galsim.__version__}')

### 6.1 Global simulation parameters

In [ ]:
# ── Telescope / detector ──────────────────────────────────────────────────────
PIXEL_SCALE  = 0.2          # arcsec / pixel  (e.g. VLT/HAWK-I)
IMAGE_SIZE   = 512          # pixels per side
PSF_FWHM     = 0.8          # arcsec  ("seeing")
PSF_SIGMA_PX = (PSF_FWHM / PIXEL_SCALE) / (2 * np.sqrt(2 * np.log(2)))
READ_NOISE   = 5.0          # electrons per pixel (rms)

# ── Sky background ────────────────────────────────────────────────────────────
SKY_LEVEL    = 200.0        # counts / pixel  (sky + dark current)

# ── Galaxy population ─────────────────────────────────────────────────────────
N_GALAXIES   = 30           # number of galaxies to simulate

print('Simulation parameters:')
print(f'  Image size      : {IMAGE_SIZE} × {IMAGE_SIZE} px')
print(f'  Pixel scale     : {PIXEL_SCALE}" / px')
print(f'  Field of view   : {IMAGE_SIZE * PIXEL_SCALE / 60:.2f} arcmin × {IMAGE_SIZE * PIXEL_SCALE / 60:.2f} arcmin')
print(f'  PSF FWHM        : {PSF_FWHM}" ({PSF_FWHM / PIXEL_SCALE:.1f} px)')
print(f'  PSF sigma       : {PSF_SIGMA_PX:.2f} px')
print(f'  Sky level       : {SKY_LEVEL} counts/px')
print(f'  Read noise      : {READ_NOISE} e- rms')
print(f'  N galaxies      : {N_GALAXIES}')

### 6.2 Helper functions

Repeated from Part 1 so this notebook is self-contained.

In [ ]:
# ── Helper functions (repeated from Part 1) ───────────────────────────────────

def b_n(n):
    """Approximation b_n ~ 2n - 0.327 (Graham & Driver 2005).
    Accurate to ~1% for n >= 1; error grows to ~6% at n = 0.5.
    """
    return 2.0 * n - 0.327

def sersic_profile(r, n, r_e=1.0, I_e=1.0):
    """Sersic profile I(r)."""
    bn = b_n(n)
    return I_e * np.exp(-bn * ((r / r_e)**(1.0 / n) - 1.0))

print('Helper functions loaded.')


---

## 9. Simulating a Single Galaxy with GalSim

GalSim draws surface-brightness profiles onto pixel grids using accurate Fourier-space methods that correctly account for pixel integration (not just point sampling). It also handles PSF convolution internally.

### Key GalSim objects used:

| Object | Description |
|--------|-------------|
| `galsim.Sersic(n, half_light_radius, flux)` | Sérsic galaxy profile |
| `galsim.Gaussian(fwhm)` | Gaussian PSF |
| `galsim.Convolve([gal, psf])` | Convolve galaxy with PSF |
| `galsim.Image(nx, ny, scale)` | Blank image canvas |
| `obj.drawImage(image)` | Render profile onto image |
| `galsim.PoissonNoise(sky_level)` | Add Poisson sky noise |
| `galsim.GaussianNoise(sigma)` | Add Gaussian read noise |

In [ ]:
# ── Define one galaxy: disk (n=1) and elliptical (n=4) ───────────────────────
gal_params = [
    dict(n=1.0, half_light_radius=0.6, flux=5000.0, q=0.6, pa=30.0,  label='Disk (n=1)'),
    dict(n=4.0, half_light_radius=0.4, flux=5000.0, q=0.8, pa=60.0,  label='Elliptical (n=4)'),
]

# Gaussian PSF
psf = galsim.Gaussian(fwhm=PSF_FWHM)  # arcsec

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for row, gp in enumerate(gal_params):
    # Build the Sérsic profile
    gal = galsim.Sersic(n=gp['n'], half_light_radius=gp['half_light_radius'],
                        flux=gp['flux'])

    # Apply ellipticity (axis ratio q and position angle)
    gal = gal.shear(q=gp['q'], beta=gp['pa'] * galsim.degrees)

    # Noiseless, PSF-free image (intrinsic profile)
    stamp_size = 101
    img_pure = galsim.Image(stamp_size, stamp_size, scale=PIXEL_SCALE)
    gal.drawImage(image=img_pure)

    # Convolve with PSF
    gal_conv = galsim.Convolve([gal, psf])
    img_conv = galsim.Image(stamp_size, stamp_size, scale=PIXEL_SCALE)
    gal_conv.drawImage(image=img_conv)

    # Add sky background and Poisson noise
    img_noisy = img_conv.copy()
    img_noisy += SKY_LEVEL
    noise = galsim.PoissonNoise(rng=galsim.BaseDeviate(42), sky_level=0.0)
    img_noisy.addNoise(noise)
    img_noisy -= SKY_LEVEL  # subtract sky so we can compare to model

    arr_pure  = img_pure.array
    arr_conv  = img_conv.array
    arr_noisy = img_noisy.array

    # Compute SNR
    snr = gp['flux'] / np.sqrt(gp['flux'] + stamp_size**2 * SKY_LEVEL + stamp_size**2 * READ_NOISE**2)

    titles = ['Intrinsic profile', f'PSF-convolved\n(FWHM={PSF_FWHM}")', 'Noisy observation']
    arrs   = [arr_pure, arr_conv, arr_noisy]

    for col, (arr, title) in enumerate(zip(arrs, titles)):
        ax = axes[row, col]
        vmax = np.percentile(arr_conv, 99.5)
        im = ax.imshow(arr, origin='lower', cmap='inferno',
                       vmin=-0.05 * vmax, vmax=vmax)
        ax.set_title(f'{gp["label"]}\n{title}', fontsize=9)
        ax.set_xlabel('x [px]')
        ax.set_ylabel('y [px]')
        plt.colorbar(im, ax=ax, label='counts')

        # Mark effective radius
        r_e_px = gp['half_light_radius'] / PIXEL_SCALE
        circle = plt.Circle((stamp_size//2, stamp_size//2), r_e_px,
                             color='cyan', fill=False, lw=1.2, ls='--', label=r'$r_e$')
        ax.add_patch(circle)

    print(f"{gp['label']:20s}  r_e={gp['half_light_radius']}"  f" arcsec  flux={gp['flux']:.0f}  SNR≈{snr:.0f}")

plt.suptitle('GalSim: Sérsic Galaxy Simulation\n(cyan dashed circle = effective radius $r_e$)', fontsize=11)
plt.tight_layout()
plt.savefig('galsim_single_galaxy.pdf', bbox_inches='tight')
plt.show()

---

## 10. Simulating a Multi-Galaxy Field

We now place $N = 30$ galaxies with randomised parameters into a 512×512 pixel field, mimicking a real survey image.

In [ ]:
# ── Generate random galaxy parameters ─────────────────────────────────────────
# All angular sizes in arcsec; positions in arcsec from image centre

half_size_arcsec = IMAGE_SIZE * PIXEL_SCALE / 2.0  # 51.2 arcsec

cat = {}
cat['x_true']   = RNG.uniform(-half_size_arcsec * 0.85, half_size_arcsec * 0.85, N_GALAXIES)
cat['y_true']   = RNG.uniform(-half_size_arcsec * 0.85, half_size_arcsec * 0.85, N_GALAXIES)
cat['n_true']   = RNG.uniform(0.5, 5.0, N_GALAXIES)         # Sérsic index
cat['r_e_true'] = RNG.uniform(0.3, 1.5, N_GALAXIES)         # arcsec
cat['flux_true']= RNG.uniform(500, 8000, N_GALAXIES)         # counts
cat['q_true']   = RNG.uniform(0.3, 1.0, N_GALAXIES)         # axis ratio
cat['pa_true']  = RNG.uniform(0, 180, N_GALAXIES)            # deg

# Convert positions to pixel coordinates (image centre = pixel 256.0)
cat['x_px'] = cat['x_true'] / PIXEL_SCALE + IMAGE_SIZE / 2
cat['y_px'] = cat['y_true'] / PIXEL_SCALE + IMAGE_SIZE / 2

input_table = Table(cat)

print(f'Input catalogue: {N_GALAXIES} galaxies')
print(f'  n   : {cat["n_true"].min():.2f} – {cat["n_true"].max():.2f}')
print(f'  r_e : {cat["r_e_true"].min():.2f} – {cat["r_e_true"].max():.2f} arcsec')
print(f'  flux: {cat["flux_true"].min():.0f} – {cat["flux_true"].max():.0f} counts')
print(f'  q   : {cat["q_true"].min():.2f} – {cat["q_true"].max():.2f}')

In [ ]:
# ── Build the full field image ────────────────────────────────────────────────
psf_galsim = galsim.Gaussian(fwhm=PSF_FWHM)

# Sky + read noise image (before adding galaxies)
full_image = galsim.Image(IMAGE_SIZE, IMAGE_SIZE, scale=PIXEL_SCALE)

for i in range(N_GALAXIES):
    # Build Sérsic profile
    gal = galsim.Sersic(
        n               = cat['n_true'][i],
        half_light_radius = cat['r_e_true'][i],
        flux            = cat['flux_true'][i],
    )
    gal = gal.shear(q=cat['q_true'][i], beta=cat['pa_true'][i] * galsim.degrees)

    # Apply sky position: shift() takes plain float values in arcsec
    gal = gal.shift(float(cat['x_true'][i]), float(cat['y_true'][i]))

    # Convolve with PSF
    obj = galsim.Convolve([gal, psf_galsim])

    # Render onto full image (add_to_image=True accumulates)
    obj.drawImage(image=full_image, add_to_image=True)

# Keep a noiseless reference
noiseless = full_image.array.copy()

# Add sky background + Poisson noise
full_image += SKY_LEVEL
full_image.addNoise(galsim.PoissonNoise(rng=galsim.BaseDeviate(1234), sky_level=0.0))

# Add Gaussian read noise
full_image.addNoise(galsim.GaussianNoise(rng=galsim.BaseDeviate(5678), sigma=READ_NOISE))

image_data = full_image.array.copy()  # numpy array, with sky

print(f'Image statistics (with sky):')
print(f'  Min  : {image_data.min():.1f} counts')
print(f'  Max  : {image_data.max():.1f} counts')
print(f'  Mean : {image_data.mean():.1f} counts  (sky ~ {SKY_LEVEL} counts)')
print(f'  Std  : {image_data.std():.1f} counts  (sky noise ~ {np.sqrt(SKY_LEVEL):.1f})')

In [ ]:
# ── Visualise the field ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Stretch using arcsinh for better visibility of faint sources
sky_sub = image_data - SKY_LEVEL  # sky-subtracted
stretch = lambda arr: np.arcsinh(arr / np.sqrt(SKY_LEVEL))

for ax, (arr, title) in zip(axes, [
    (noiseless, 'Noiseless model field'),
    (sky_sub,   'Noisy observation (sky-subtracted)'),
]):
    vmin, vmax = np.percentile(stretch(sky_sub), [0.5, 99.8])
    im = ax.imshow(stretch(arr), origin='lower', cmap='afmhot', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label=r'arcsinh$(S / \sqrt{S_{\rm sky}})$')

    # Mark input galaxy positions
    ax.scatter(cat['x_px'], cat['y_px'], marker='+', color='cyan',
               s=60, lw=1.2, label='True positions')
    ax.set_xlabel('x [px]')
    ax.set_ylabel('y [px]')
    ax.set_title(title)

axes[1].legend(fontsize=9)

plt.suptitle(f'Simulated galaxy field — {N_GALAXIES} galaxies  '
             f'({IMAGE_SIZE}×{IMAGE_SIZE} px, scale={PIXEL_SCALE}"/px, '
             f'PSF FWHM={PSF_FWHM}")', fontsize=10)
plt.tight_layout()
plt.savefig('galsim_field.pdf', bbox_inches='tight')
plt.show()

---

## 11. Source Detection with photutils

We follow the standard pipeline:
1. **Estimate and subtract the background** using a 2D background map.
2. **Detect connected regions** above $2.5\,\sigma_\text{bg}$ covering at least 10 pixels.
3. **Build a source catalogue** with positions, fluxes, and shape parameters.
4. **Cross-match** with the input catalogue to assess completeness and purity.

In [ ]:
# ── Step 1: Background estimation ────────────────────────────────────────────
# Divide image into a grid of boxes; estimate median in each box.
# box_size should be larger than typical sources but smaller than any
# large-scale sky gradient.

bkg_estimator = MedianBackground()
bkg = Background2D(
    image_data,
    box_size    = 64,    # grid box size in pixels
    filter_size = 3,     # smooth the background map with a 3×3 median filter
    bkg_estimator = bkg_estimator,
)

bkg_level = bkg.background          # 2D background map (counts)
bkg_rms   = bkg.background_rms      # 2D background RMS map (counts)

# Sky-subtracted image
image_sub = image_data - bkg_level

print(f'Background estimation:')
print(f'  Mean background level : {bkg_level.mean():.1f} counts  (true sky = {SKY_LEVEL})')
print(f'  Mean background RMS   : {bkg_rms.mean():.1f} counts  (expected ≈ {np.sqrt(SKY_LEVEL):.1f})')
print(f'  Sky-subtracted image mean: {image_sub.mean():.2f} counts  (should ≈ 0)')

In [ ]:
# ── Step 2: Source detection ──────────────────────────────────────────────────
# Detect pixels > threshold = 2.5 × bkg_rms, in connected groups of ≥ 10 px.

DETECT_THRESH  = 2.5   # sigma above background
MIN_AREA_PX    = 10    # minimum source area in pixels

threshold = DETECT_THRESH * bkg_rms

segm = detect_sources(
    image_sub,
    threshold = threshold,
    npixels   = MIN_AREA_PX,
)

print(f'Detection threshold: {DETECT_THRESH}σ above background')
print(f'Minimum source area: {MIN_AREA_PX} pixels')
print(f'Number of detected sources: {segm.nlabels}')
print(f'Input galaxies             : {N_GALAXIES}')

In [ ]:
# ── Step 3: Source catalogue ──────────────────────────────────────────────────
cat_det = SourceCatalog(image_sub, segm, background=bkg_level)

# Extract key measurements (photutils ≥ 1.0 attribute names)
x_det    = cat_det.x_centroid              # x pixel centroid
y_det    = cat_det.y_centroid              # y pixel centroid
flux_det = cat_det.segment_flux            # sum of pixels in segment (counts)
a_det    = cat_det.semimajor_axis.value    # 1-sigma semi-major axis [px] (2nd moment)
b_det    = cat_det.semiminor_axis.value    # 1-sigma semi-minor axis [px]

print(f'Detected source properties:')
print(f'  x_centroid: {x_det.min():.1f} – {x_det.max():.1f} px')
print(f'  y_centroid: {y_det.min():.1f} – {y_det.max():.1f} px')
print(f'  Flux      : {flux_det.min():.0f} – {flux_det.max():.0f} counts')
print(f'  Semi-major: {a_det.min():.2f} – {a_det.max():.2f} px')

In [ ]:
# ── Figure: segmentation map + detections overlaid ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sky-subtracted image
ax = axes[0]
vmin, vmax = np.percentile(image_sub, [1, 99.5])
ax.imshow(image_sub, origin='lower', cmap='afmhot', vmin=vmin, vmax=vmax)
ax.scatter(x_det, y_det, marker='o', facecolors='none', edgecolors='cyan',
           s=80, lw=1.5, label=f'{segm.nlabels} detected')
ax.scatter(cat['x_px'], cat['y_px'], marker='+', color='yellow',
           s=60, lw=1.2, label=f'{N_GALAXIES} input')
ax.set_title('Sky-subtracted image')
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')
ax.legend(fontsize=9)

# Segmentation map
ax = axes[1]
ax.imshow(segm.data, origin='lower', cmap='nipy_spectral',
          vmin=0, vmax=segm.nlabels)
ax.scatter(x_det, y_det, marker='+', color='white', s=40, lw=1.2)
ax.set_title(f'Segmentation map ({segm.nlabels} objects)')
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')

plt.suptitle('photutils Source Detection', fontsize=11)
plt.tight_layout()
plt.savefig('galsim_detection.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Cross-match: nearest detected source to each input galaxy ─────────────────
from scipy.spatial import cKDTree

MATCH_RADIUS_PX = 5.0  # maximum match radius in pixels

det_coords = np.column_stack([x_det, y_det])
inp_coords = np.column_stack([cat['x_px'], cat['y_px']])

tree = cKDTree(det_coords)
dists, idx_match = tree.query(inp_coords, distance_upper_bound=MATCH_RADIUS_PX)

matched = dists < MATCH_RADIUS_PX
n_matched = matched.sum()

print(f'Cross-match results (match radius = {MATCH_RADIUS_PX} px):')
print(f'  Input galaxies        : {N_GALAXIES}')
print(f'  Detected sources      : {segm.nlabels}')
print(f'  Matched (input→det)   : {n_matched}')
print(f'  Completeness          : {n_matched / N_GALAXIES * 100:.0f}%')
print(f'  Missed galaxies       : {N_GALAXIES - n_matched}')

---

## 12. Galaxy Profile Fitting

For each matched detection we:
1. Cut a small **postage stamp** (48×48 px) around the galaxy.
2. Build an initial guess for the Sérsic parameters from the segmentation catalogue.
3. Fit `astropy.modeling.models.Sersic2D` using Levenberg–Marquardt least squares.
4. Store the best-fit parameters and compare to the truth.

> **Note:** We fit in image space without PSF deconvolution here. This biases $r_e$ slightly upward for the most compact galaxies. In research one would convolve the model with the PSF before comparing to the data (as GALFIT does).

In [ ]:
from astropy.modeling.models import Sersic2D

STAMP_HALF = 24  # half-size of fitting stamp in pixels

results = []

fitter = fitting.LevMarLSQFitter()

for i in range(N_GALAXIES):
    if not matched[i]:
        continue

    # Centroid from detection catalogue
    j_det = idx_match[i]
    xc = x_det[j_det]
    yc = y_det[j_det]

    # Cut a stamp from sky-subtracted image
    xi0 = int(np.round(xc)) - STAMP_HALF
    xi1 = int(np.round(xc)) + STAMP_HALF
    yi0 = int(np.round(yc)) - STAMP_HALF
    yi1 = int(np.round(yc)) + STAMP_HALF

    # Skip stamps that go outside the image
    if xi0 < 0 or yi0 < 0 or xi1 >= IMAGE_SIZE or yi1 >= IMAGE_SIZE:
        continue

    stamp = image_sub[yi0:yi1, xi0:xi1].copy()
    stamp_size_y, stamp_size_x = stamp.shape

    # Pixel coordinate grids (relative to stamp)
    yy, xx = np.mgrid[0:stamp_size_y, 0:stamp_size_x]
    xc_stamp = xc - xi0
    yc_stamp = yc - yi0

    # Noise map for weighting
    sigma_map = np.sqrt(np.abs(stamp) + SKY_LEVEL + READ_NOISE**2)

    # Initial guess: take r_e from second-moment size, n=2 (neutral)
    r_e_guess = max(a_det[j_det] * PIXEL_SCALE, 0.3)  # arcsec
    r_e_guess_px = r_e_guess / PIXEL_SCALE

    model_init = Sersic2D(
        amplitude    = stamp.max(),
        r_eff        = r_e_guess_px,
        n            = 2.0,
        x_0          = xc_stamp,
        y_0          = yc_stamp,
        ellip        = 1 - b_det[j_det] / max(a_det[j_det], 0.5),
        theta        = 0.0,
        bounds={
            'n':         (0.3, 8.0),
            'r_eff':     (0.5, STAMP_HALF * 0.9),
            'ellip':     (0.0, 0.95),
            'amplitude': (0.0, None),
        }
    )

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            model_fit = fitter(model_init, xx, yy, stamp,
                               weights=1.0 / sigma_map, maxiter=500)

        results.append({
            'i_input'   : i,
            'n_in'      : cat['n_true'][i],
            'r_e_in'    : cat['r_e_true'][i],
            'flux_in'   : cat['flux_true'][i],
            'q_in'      : cat['q_true'][i],
            'n_out'     : model_fit.n.value,
            'r_e_out'   : model_fit.r_eff.value * PIXEL_SCALE,  # → arcsec
            'flux_out'  : flux_det[j_det],
            'q_out'     : 1 - model_fit.ellip.value,
            'model_fit' : model_fit,
            'stamp'     : stamp,
            'xi0'       : xi0, 'yi0': yi0,
            'xc_stamp'  : xc_stamp, 'yc_stamp': yc_stamp,
        })
    except Exception:
        pass

print(f'Successfully fitted: {len(results)} / {n_matched} matched galaxies')

In [ ]:
# ── Figure: data / model / residual for 4 galaxies ───────────────────────────
N_SHOW = min(4, len(results))
fig, axes = plt.subplots(N_SHOW, 3, figsize=(11, 3 * N_SHOW))

for row, res in enumerate(results[:N_SHOW]):
    stamp = res['stamp']
    mfit  = res['model_fit']
    yy_s, xx_s = np.mgrid[0:stamp.shape[0], 0:stamp.shape[1]]
    model_arr   = mfit(xx_s, yy_s)
    resid_arr   = stamp - model_arr

    vmax = np.percentile(stamp, 99)
    vmin_r, vmax_r = np.percentile(resid_arr, [2, 98])

    for col, (arr, title, cmap, vlo, vhi) in enumerate([
        (stamp,      'Data',     'inferno',  -0.05*vmax, vmax),
        (model_arr,  'Model',    'inferno',  -0.05*vmax, vmax),
        (resid_arr,  'Residual', 'RdBu_r',   vmin_r,     vmax_r),
    ]):
        ax = axes[row, col]
        im = ax.imshow(arr, origin='lower', cmap=cmap, vmin=vlo, vmax=vhi)
        plt.colorbar(im, ax=ax, label='counts')
        snr_gal = res['flux_in'] / np.sqrt(res['flux_in'] + stamp.size * SKY_LEVEL)
        ax.set_title(f'{title}  (galaxy #{res["i_input"]+1}, SNR≈{snr_gal:.0f})', fontsize=8)
        ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]')

    # Print fit parameters for this galaxy
    print(f'Galaxy {res["i_input"]+1:2d}:  '
          f'n_in={res["n_in"]:.2f} → n_out={res["n_out"]:.2f}   '
          f'r_e_in={res["r_e_in"]:.2f}" → r_e_out={res["r_e_out"]:.2f}"')

plt.suptitle('Sérsic Profile Fitting: Data | Model | Residual', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('galsim_fitting_triptych.pdf', bbox_inches='tight')
plt.show()

---

## 13. Input–Output Comparison

We now compare the **recovered** parameters to the **true** input values. A perfect recovery would lie on the $y = x$ (one-to-one) line. Scatter around this line quantifies the **measurement uncertainty**; systematic offset is **bias**.

In [ ]:
if len(results) < 3:
    print('Too few fitted galaxies for statistics — adjust detection parameters.')
else:
    n_in    = np.array([r['n_in']    for r in results])
    n_out   = np.array([r['n_out']   for r in results])
    re_in   = np.array([r['r_e_in']  for r in results])
    re_out  = np.array([r['r_e_out'] for r in results])
    fi      = np.array([r['flux_in'] for r in results])
    fo      = np.array([r['flux_out']for r in results])

    # Per-galaxy SNR (rough estimate: total signal / sqrt(sky in stamp))
    snr_arr = fi / np.sqrt(fi + (2 * STAMP_HALF)**2 * SKY_LEVEL)

    # ── Figure: 3 panels ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    panel_data = [
        (n_in,   n_out,  [0.4, 6], r'Sérsic index $n$'),
        (re_in,  re_out, [0.2, 2], r'Effective radius $r_e$ [arcsec]'),
        (fi,     fo,     [0, 9000], 'Flux [counts]'),
    ]

    for ax, (x, y, lim, lbl) in zip(axes, panel_data):
        sc = ax.scatter(x, y, c=np.log10(snr_arr + 1), cmap='viridis',
                        s=50, edgecolors='k', lw=0.5, zorder=3)
        ax.plot(lim, lim, 'r--', lw=1.5, label='1:1 line')
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel(f'Input {lbl}')
        ax.set_ylabel(f'Output {lbl}')

        # Bias and scatter
        delta = y - x
        ax.set_title(f'{lbl}\n'
                     f'bias = {np.median(delta):.3f},  '
                     f'scatter = {np.std(delta):.3f}', fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.colorbar(sc, ax=axes, label=r'$\log_{10}(\mathrm{SNR})$', shrink=0.8)
    plt.suptitle('Input vs Output Parameter Recovery', fontsize=11)
    plt.tight_layout()
    plt.savefig('galsim_io_comparison.pdf', bbox_inches='tight')
    plt.show()

    # Summary table
    print('\nParameter recovery summary:')
    print(f'{"Parameter":<18} {"Median bias":>14} {"RMS scatter":>14}')
    print('-' * 50)
    for name, x, y in [('Sérsic n', n_in, n_out),
                        ('r_e [arcsec]', re_in, re_out),
                        ('Flux [counts]', fi, fo)]:
        d = y - x
        print(f'{name:<18} {np.median(d):>+14.3f} {np.std(d):>14.3f}')

---

## 14. Effect of SNR on Parameter Recovery

We expect recovery to degrade for faint (low-SNR) galaxies. Here we bin the results by SNR and show how bias and scatter evolve.

In [ ]:
if len(results) < 5:
    print('Too few fitted galaxies for SNR binning.')
else:
    snr_bins = [0, 10, 20, 50, 1e6]
    bin_labels = ['SNR < 10', '10–20', '20–50', 'SNR > 50']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, (param_name, x_arr, y_arr) in zip(
        axes,
        [('Sérsic $n$', n_in, n_out), (r'$r_e$ [arcsec]', re_in, re_out)]
    ):
        fractional = (y_arr - x_arr) / x_arr
        bin_data = []
        for lo, hi in zip(snr_bins[:-1], snr_bins[1:]):
            mask = (snr_arr >= lo) & (snr_arr < hi)
            if mask.sum() > 0:
                bin_data.append(fractional[mask])
            else:
                bin_data.append(np.array([np.nan]))

        bp = ax.boxplot(bin_data, labels=bin_labels, patch_artist=True,
                        medianprops={'color': 'red', 'lw': 2})
        for patch in bp['boxes']:
            patch.set_facecolor('steelblue')
            patch.set_alpha(0.5)
        ax.axhline(0, color='k', lw=1, ls='--')
        ax.set_xlabel('SNR bin')
        ax.set_ylabel(r'$(\theta_\mathrm{out} - \theta_\mathrm{in}) / \theta_\mathrm{in}$')
        ax.set_title(f'Fractional error in {param_name}')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-1.5, 1.5)

    plt.suptitle('Parameter Recovery vs SNR', fontsize=11)
    plt.tight_layout()
    plt.savefig('galsim_snr_recovery.pdf', bbox_inches='tight')
    plt.show()

    print('Observation: scatter grows and bias increases for low-SNR galaxies.')
    print('n is the hardest parameter to recover (it controls the profile shape).')

---

## 15. Visualising the PSF Effect

A key insight: the PSF broadens profiles, making small galaxies harder to characterise. Here we show how much the PSF changes the observed profile compared to the true profile.

In [ ]:
# -- True (intrinsic) profile: analytic Sersic formula -----------------------
# -- PSF-convolved profile: GalSim rendering ----------------------------------
# Note: analytic formula is used for the true profile to avoid GalSim FFT
#       issues when rendering compact profiles without PSF convolution.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (r_e_val, n_val, title) in zip(axes, [
    (0.3, 2.0, r'Small galaxy ($r_e = 0.3"$, $n=2$)'),
    (1.5, 2.0, r'Large galaxy ($r_e = 1.5"$, $n=2$)'),
]):
    # Analytic true profile (sersic_profile defined in section 7)
    r_max  = max(4.0, 5.0 * r_e_val)
    r_arr  = np.linspace(0.001, r_max, 500)
    I_true = sersic_profile(r_arr, n_val, r_e=r_e_val)
    I_true = I_true / I_true.max()

    # PSF-convolved profile via GalSim
    stamp_sz = 201
    gal_gs   = galsim.Sersic(n=n_val, half_light_radius=r_e_val, flux=1e5)
    conv_gs  = galsim.Convolve([gal_gs, galsim.Gaussian(fwhm=PSF_FWHM)])
    img_c    = galsim.Image(stamp_sz, stamp_sz, scale=PIXEL_SCALE)
    conv_gs.drawImage(image=img_c)

    cx       = stamp_sz // 2
    row_conv = img_c.array[cx, :]
    x_arcsec = (np.arange(stamp_sz) - cx) * PIXEL_SCALE
    mask     = (x_arcsec >= 0) & (x_arcsec <= r_max)
    peak     = max(row_conv.max(), 1e-30)

    ax.plot(r_arr,           I_true,              'b-', lw=2, label='True Sersic (analytic)')
    ax.plot(x_arcsec[mask],  row_conv[mask]/peak, 'r-', lw=2, label='PSF-convolved (GalSim)')
    ax.axvline(r_e_val,      color='gray',  ls='--', lw=1,   label=f'$r_e = {r_e_val}"$')
    ax.axvline(PSF_FWHM / 2, color='green', ls=':',  lw=1.2, label=f'PSF HWHM = {PSF_FWHM/2}"')
    ax.set_xlabel('Radius [arcsec]')
    ax.set_ylabel('Normalised surface brightness')
    ax.set_title(title)
    ax.set_yscale('log')
    ax.set_ylim(1e-4, 2)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'PSF Effect on Observed Profiles  (PSF FWHM = {PSF_FWHM}")', fontsize=11)
plt.tight_layout()
plt.savefig('galsim_psf_effect.pdf', bbox_inches='tight')
plt.show()

print('Key result: the small galaxy (r_e < PSF HWHM) is essentially unresolved.')
print('Its observed profile is dominated by the PSF shape, not the galaxy shape.')


---

## 16. Professional Tools: GALFIT and GALFITools

The approach in this notebook (Sérsic fit with `astropy.modeling`) is instructive but simplified. In research, the standard tool is **GALFIT** (Peng et al. 2002, 2010), which:

- Fits **multiple Sérsic components** simultaneously (bulge + disk + bar + …).
- **Convolves the model with the PSF** at every iteration (essential for compact galaxies).
- Supports many more profile types: Nuker, Gaussian, Moffat, Exponential, Ferrer, King, …
- Provides per-parameter uncertainties from the covariance matrix.

[**GALFITools**](https://pypi.org/project/GALFITools/) (C. Añorve) is a Python package that wraps GALFIT: it creates the input `.feedme` configuration files, runs GALFIT, reads the output `.fits` files, and provides analysis and mask-creation utilities.

```python
# Example workflow (requires GALFIT binary installed on your system):
# pip install GALFITools

# from galfittools import GALFITools
# gft = GALFITools()
# gft.RunGALFIT(feedme='galaxy.feedme')
# result = gft.ReadOutput('galaxy_out.fits')
# print(result.sersic.n, result.sersic.re)
```

GALFIT itself (compiled binary, free for academic use) must be downloaded separately from [https://users.obs.carnegiescience.edu/peng/work/galfit/galfit.html](https://users.obs.carnegiescience.edu/peng/work/galfit/galfit.html).

---

## 17. Summary

### Key concepts

| Concept | Formula / tool | Take-away |
|---------|---------------|----------|
| Sérsic profile | $I(r) = I_e \exp\{-b_n[(r/r_e)^{1/n}-1]\}$ | $n, r_e, I_e$ fully describe a galaxy's brightness profile |
| PSF convolution | $I_\text{obs} = I_\text{true} * \text{PSF}$ | Atmospheric blurring smears and broadens the profile |
| Image noise | $\sigma^2 = N_\text{gal} + N_\text{sky} + \sigma^2_\text{read}$ | Noise limits the precision of parameter recovery |
| Detection | `photutils.detect_sources` | Connected-component labelling above $k\sigma$ threshold |
| Profile fitting | $\chi^2 = \sum (I - M)^2/\sigma^2$ | Levenberg–Marquardt minimisation via `astropy.modeling` |
| Parameter recovery | scatter($n$) $>$ scatter($r_e$) | Shape ($n$) is harder to recover than size ($r_e$) |

### Workflow recap

```
GalSim: define Sérsic → shear → shift → convolve PSF → draw pixels → add noise
   ↓
photutils: estimate background → subtract → detect sources → build catalogue
   ↓
astropy.modeling: cut stamp → fit Sersic2D → residuals → compare to truth
```

### Further reading

- **GalSim documentation:** [https://galsim-developers.github.io/GalSim/_build/html/index.html](https://galsim-developers.github.io/GalSim/_build/html/index.html)
- **photutils documentation:** [https://photutils.readthedocs.io/en/stable/](https://photutils.readthedocs.io/en/stable/)
- **Sérsic profiles review:** Graham & Driver (2005), PASA 22, 118 — [arXiv:astro-ph/0503176](https://arxiv.org/abs/astro-ph/0503176)
- **GALFIT paper:** Peng et al. (2002), AJ 124, 266 — [arXiv:astro-ph/0204182](https://arxiv.org/abs/astro-ph/0204182)
- **GalSim paper:** Rowe et al. (2015), A&C 10, 121 — [arXiv:1407.7676](https://arxiv.org/abs/1407.7676)

---

## 18. Exercises

1. **PSF size effect** *(easy)*: Change `PSF_FWHM` from 0.8" to 0.3" (sharper seeing) and 2.0" (poor seeing). How does completeness and parameter recovery change? Which galaxies become unresolved?

2. **Sérsic index recovery** *(medium)*: Fix all galaxy parameters to $n = 1$ (pure disks) and re-run the simulation and fitting. Plot the distribution of recovered $n_\text{out}$ values. What does the spread tell you about the measurement uncertainty?

3. **Background subtraction failure** *(medium)*: Simulate a strong sky gradient by adding `0.5 * np.arange(IMAGE_SIZE)` to each row of the image before detection. How does this affect the number of detections and the flux measurements? Fix it by adjusting the background estimation parameters.

4. **Deblending** *(hard)*: Simulate two galaxies with overlapping profiles (place them within 5 arcsec of each other). The standard `detect_sources` will likely merge them into one object. Look at the `photutils.segmentation.deblend_sources` function and use it to separate the pair. Check whether the fit improves.

5. **PSF-corrected fitting** *(hard)*: Modify the fitting loop to convolve the Sérsic model with the PSF kernel (`astropy.convolution.convolve_fft`) before comparing to the data. Quantify how much this reduces the bias in $r_e$ for the most compact galaxies ($r_e < 0.5''$).